# Image Classification with PyTorch — Assignment II

## Approach Overview

This notebook implements a 10-class image classification pipeline using PyTorch. The approach is as follows:

1. **Custom Dataset**: A `torch.utils.data.Dataset` subclass loads images from disk using ids in `Train.csv` / `Test.csv`.
2. **Data Augmentation**: Training images are augmented with random flips, rotations, colour jitter, and random erasing to reduce overfitting and improve generalisation.
3. **Transfer Learning (EfficientNet-B2)**: We fine-tune a pre-trained EfficientNet-B2 backbone — state-of-the-art accuracy at modest compute cost.
4. **Hyper-parameter Optimisation with Optuna**: We use Optuna (Tree-structured Parzen Estimator — TPE) to search over learning rate, weight decay, dropout rate, and batch size automatically.
5. **Training**: We use AdamW + CosineAnnealingLR, with early stopping to avoid overfitting.
6. **Prediction**: The best checkpoint is loaded, and predictions are generated for every id in `Test.csv`.

### Why EfficientNet-B2?
EfficientNet models scale width, depth, and resolution in a principled way. B2 provides a sweet spot between accuracy and speed, and ImageNet pre-training gives powerful feature representations that transfer well to custom datasets, especially when labelled data is limited.

### Why Optuna (TPE)?
Grid search is exponential; random search ignores history. Optuna's TPE algorithm builds a probabilistic model of which hyper-parameter regions yielded good results and samples new candidates from promising areas — typically converging to a good solution in far fewer trials than grid/random search.

## 1. Install Dependencies

In [ ]:
# Install required packages (run once)
import subprocess, sys
pkgs = ['optuna', 'timm', 'torchvision', 'pandas', 'pillow', 'scikit-learn', 'matplotlib', 'tqdm']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
print('All packages ready.')

## 2. Imports & Configuration

In [ ]:
import os, random, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
import timm   # provides EfficientNet and many other pretrained models
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── Paths — adjust if your layout differs ────────────────────────────────────
DATA_DIR   = Path('dataset')          # root folder after unzipping
IMG_DIR    = DATA_DIR / 'images'      # folder containing all images
TRAIN_CSV  = DATA_DIR / 'Train.csv'
TEST_CSV   = DATA_DIR / 'Test.csv'
CKPT_PATH  = Path('best_model.pth')   # where we save the best weights
OUTPUT_CSV = Path('Test_predictions.csv')

## 3. Exploratory Data Analysis

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print('Train shape:', train_df.shape)
print('Test  shape:', test_df.shape)
print('\nTrain columns:', train_df.columns.tolist())
print('Test  columns:', test_df.columns.tolist())
train_df.head()

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(10, 4))
train_df['label'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='k')
ax.set_title('Training Set — Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

NUM_CLASSES = train_df['label'].nunique()
print(f'Number of classes: {NUM_CLASSES}')
print(train_df['label'].value_counts())

In [ ]:
# Show sample images (one per class)
# Determine which column holds image ids
id_col = [c for c in train_df.columns if 'id' in c.lower() or 'image' in c.lower()][0]
label_col = 'label'

def get_img_path(img_id):
    """Try common extensions; return the one that exists."""
    for ext in ['', '.jpg', '.jpeg', '.png', '.bmp']:
        p = IMG_DIR / f'{img_id}{ext}'
        if p.exists():
            return p
    # fallback: list directory and match prefix
    matches = list(IMG_DIR.glob(f'{img_id}*'))
    return matches[0] if matches else None

classes = sorted(train_df[label_col].unique())
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, cls in zip(axes.flat, classes):
    row = train_df[train_df[label_col] == cls].iloc[0]
    path = get_img_path(row[id_col])
    if path:
        img = Image.open(path).convert('RGB')
        ax.imshow(img)
    ax.set_title(f'Class {cls}')
    ax.axis('off')
plt.suptitle('Sample Images per Class', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Custom Dataset

`ImageDataset` subclasses `torch.utils.data.Dataset`, implementing `__len__` and `__getitem__`. It lazily loads images from disk (memory-efficient) and applies a `transform` pipeline at read time.

In [ ]:
class ImageDataset(Dataset):
    """
    Custom Dataset that subclasses torch.utils.data.Dataset.

    Parameters
    ----------
    df        : pd.DataFrame with at minimum an id column; optionally a label column.
    img_dir   : pathlib.Path pointing to the folder that contains the image files.
    transform : torchvision transform pipeline applied to every image.
    id_col    : name of the column that holds the image filename/id.
    label_col : name of the label column; pass None for the test set.
    """

    def __init__(self, df, img_dir, transform=None, id_col='id', label_col='label'):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = Path(img_dir)
        self.transform = transform
        self.id_col    = id_col
        self.label_col = label_col
        self.has_label = label_col in df.columns

        # Build a label-to-index mapping for string labels
        if self.has_label:
            unique_labels = sorted(df[label_col].unique())
            self.label2idx = {l: i for i, l in enumerate(unique_labels)}
            self.idx2label = {i: l for l, i in self.label2idx.items()}
        else:
            self.label2idx = {}
            self.idx2label = {}

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, img_id):
        """Return the image path, trying common extensions."""
        for ext in ['', '.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']:
            p = self.img_dir / f'{img_id}{ext}'
            if p.exists():
                return p
        matches = list(self.img_dir.glob(f'{img_id}*'))
        if matches:
            return matches[0]
        raise FileNotFoundError(f'Image not found for id: {img_id} in {self.img_dir}')

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img_id = row[self.id_col]
        path   = self._resolve_path(img_id)

        image  = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        if self.has_label:
            raw_label = row[self.label_col]
            # Support both integer and string labels
            label = self.label2idx.get(raw_label, int(raw_label))
            return image, label
        return image, img_id  # return id for test-set inference

print('ImageDataset class defined ✓')

## 5. Data Augmentation

We apply a rich augmentation pipeline **only to the training set** to reduce overfitting:

| Transform | Purpose |
|---|---|
| `RandomResizedCrop` | Forces the model to handle objects at different scales/positions |
| `RandomHorizontalFlip` | Doubles effective dataset size; handles left–right symmetry |
| `RandomVerticalFlip` | Useful when orientation is ambiguous |
| `RandomRotation(15°)` | Rotational invariance |
| `ColorJitter` | Robustness to lighting and colour changes |
| `RandomGrayscale` | Prevents over-reliance on colour |
| `GaussianBlur` | Simulates camera defocus, improves texture robustness |
| `RandomErasing` | Occlusion regularisation (similar to CutOut) |
| `Normalize(ImageNet)` | Aligns statistics with the pre-trained backbone's expectations |

The **validation / test** transform uses only `Resize` + `CenterCrop` + `Normalize` — no stochastic ops — to give stable, reproducible scores.

In [ ]:
IMG_SIZE = 260  # EfficientNet-B2 native resolution

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomGrayscale(p=0.05),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.2)),
])

val_transform = T.Compose([
    T.Resize(IMG_SIZE + 20),          # slightly larger then crop
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Transforms defined ✓')

## 6. Model Definition

We use **EfficientNet-B2** from `timm`. The pre-trained classifier head is replaced with:
```
AdaptiveAvgPool → Dropout → Linear(num_features → NUM_CLASSES)
```
We **freeze the backbone** for the first few epochs (feature extraction), then unfreeze all layers for **fine-tuning** at a lower learning rate. This two-stage strategy reduces the risk of catastrophic forgetting.

In [ ]:
def build_model(num_classes: int, dropout: float = 0.3, pretrained: bool = True):
    """
    Build an EfficientNet-B2 model with a custom classification head.

    Parameters
    ----------
    num_classes : number of output classes
    dropout     : dropout probability before the final linear layer
    pretrained  : whether to load ImageNet weights
    """
    model = timm.create_model(
        'efficientnet_b2',
        pretrained=pretrained,
        num_classes=num_classes,
        drop_rate=dropout,
    )
    return model


def freeze_backbone(model):
    """Freeze everything except the classifier head."""
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False


def unfreeze_all(model):
    """Unfreeze all parameters for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True


# Quick sanity check
test_model = build_model(NUM_CLASSES)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
out   = test_model(dummy)
print(f'Model output shape: {out.shape}  (expected: [2, {NUM_CLASSES}])  ✓')
del test_model

## 7. Training & Validation Helpers

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


def train_model(
    model, train_loader, val_loader,
    lr=1e-3, weight_decay=1e-4,
    epochs=20, freeze_epochs=3,
    device=DEVICE, ckpt_path=CKPT_PATH,
    patience=5, verbose=True
):
    """
    Full training loop with:
    - Initial backbone freeze for `freeze_epochs` then full fine-tuning
    - AdamW optimiser
    - CosineAnnealingLR scheduler
    - Early stopping (patience-based)
    - Best checkpoint saving
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr / 50)

    best_val_acc = 0.0
    no_improve   = 0
    history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    # Stage 1: freeze backbone
    freeze_backbone(model)

    for epoch in range(1, epochs + 1):
        # Unfreeze at epoch freeze_epochs+1
        if epoch == freeze_epochs + 1:
            if verbose:
                print(f'\n--- Unfreezing backbone at epoch {epoch} ---')
            unfreeze_all(model)
            # Lower LR for fine-tuning
            for pg in optimizer.param_groups:
                pg['lr'] = lr / 10

        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        elapsed = time.time() - t0
        if verbose:
            print(f'Epoch {epoch:3d}/{epochs} | '
                  f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '
                  f'{elapsed:.1f}s')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f'Early stopping at epoch {epoch}.')
                break

    print(f'\nBest validation accuracy: {best_val_acc:.4f}')
    return history, best_val_acc


print('Training helpers defined ✓')

## 8. Hyper-parameter Optimisation with Optuna (TPE)

**Optuna** uses the **Tree-structured Parzen Estimator (TPE)** algorithm, a Bayesian optimisation method. It maintains two density estimators — one for good configurations and one for bad ones — and samples from the ratio that maximises the Expected Improvement (EI). This is far more sample-efficient than grid/random search.

We search over:
| Hyper-parameter | Range |
|---|---|
| Learning rate | 1e-4 – 1e-2 (log scale) |
| Weight decay | 1e-5 – 1e-2 (log scale) |
| Dropout | 0.1 – 0.5 |
| Batch size | {16, 32, 64} |

Each trial trains for **5 epochs** on a 80/20 stratified split, trading search speed for quality. After the study, we retrain the best config for more epochs.

In [ ]:
# Identify the id and label columns (auto-detect)
id_col    = [c for c in train_df.columns if 'id' in c.lower() or 'image' in c.lower()][0]
label_col = 'label'
print(f'id column: "{id_col}", label column: "{label_col}"')

# Build label encoder from training data
full_train_ds = ImageDataset(train_df, IMG_DIR, transform=train_transform,
                              id_col=id_col, label_col=label_col)
LABEL2IDX = full_train_ds.label2idx
IDX2LABEL = full_train_ds.idx2label
print('Label mapping:', LABEL2IDX)

In [ ]:
from sklearn.model_selection import train_test_split

labels_arr = train_df[label_col].values
train_idx, val_idx = train_test_split(
    np.arange(len(train_df)), test_size=0.2,
    stratify=labels_arr, random_state=SEED
)
print(f'Train split: {len(train_idx)} | Val split: {len(val_idx)}')


def objective(trial):
    """Optuna objective: returns validation accuracy for the given hyper-params."""
    lr           = trial.suggest_float('lr',           1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)
    dropout      = trial.suggest_float('dropout',      0.1,  0.5)
    batch_size   = trial.suggest_categorical('batch_size', [16, 32, 64])

    # Datasets
    tr_ds = ImageDataset(train_df.iloc[train_idx], IMG_DIR, train_transform, id_col, label_col)
    tr_ds.label2idx = LABEL2IDX; tr_ds.idx2label = IDX2LABEL
    vl_ds = ImageDataset(train_df.iloc[val_idx],   IMG_DIR, val_transform,   id_col, label_col)
    vl_ds.label2idx = LABEL2IDX; vl_ds.idx2label = IDX2LABEL

    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    vl_loader = DataLoader(vl_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = build_model(NUM_CLASSES, dropout=dropout).to(DEVICE)
    _, val_acc = train_model(
        model, tr_loader, vl_loader,
        lr=lr, weight_decay=weight_decay,
        epochs=5, freeze_epochs=1,
        device=DEVICE, ckpt_path=Path(f'trial_{trial.number}.pth'),
        patience=5, verbose=False
    )
    # Clean up trial checkpoint
    Path(f'trial_{trial.number}.pth').unlink(missing_ok=True)
    return val_acc


N_TRIALS = 15   # increase for better search at the cost of time
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
print(f'Starting Optuna hyper-parameter search ({N_TRIALS} trials) …')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
print(f'\nBest trial: {study.best_trial.number}')
print(f'Best validation accuracy: {study.best_value:.4f}')
print('Best hyper-parameters:', best_params)

In [ ]:
# Visualise Optuna optimisation history
trial_nums  = [t.number for t in study.trials]
trial_vals  = [t.value  for t in study.trials]
running_best = np.maximum.accumulate(trial_vals)

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(trial_nums, trial_vals, s=40, alpha=0.7, label='Trial accuracy')
ax.plot(trial_nums, running_best, color='tomato', lw=2, label='Best so far')
ax.set_xlabel('Trial')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Optuna TPE Hyper-parameter Search')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Full Training with Best Hyper-parameters

In [ ]:
BEST_LR      = best_params['lr']
BEST_WD      = best_params['weight_decay']
BEST_DROPOUT = best_params['dropout']
BEST_BATCH   = best_params['batch_size']
FREEZE_EPOCHS = 3
TOTAL_EPOCHS  = 40
PATIENCE      = 8

# Re-create datasets with the full split
tr_ds = ImageDataset(train_df.iloc[train_idx], IMG_DIR, train_transform, id_col, label_col)
tr_ds.label2idx = LABEL2IDX; tr_ds.idx2label = IDX2LABEL
vl_ds = ImageDataset(train_df.iloc[val_idx],   IMG_DIR, val_transform,   id_col, label_col)
vl_ds.label2idx = LABEL2IDX; vl_ds.idx2label = IDX2LABEL

tr_loader = DataLoader(tr_ds, batch_size=BEST_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
vl_loader = DataLoader(vl_ds, batch_size=BEST_BATCH, shuffle=False, num_workers=2, pin_memory=True)

final_model = build_model(NUM_CLASSES, dropout=BEST_DROPOUT).to(DEVICE)

history, best_val_acc = train_model(
    final_model, tr_loader, vl_loader,
    lr=BEST_LR, weight_decay=BEST_WD,
    epochs=TOTAL_EPOCHS, freeze_epochs=FREEZE_EPOCHS,
    device=DEVICE, ckpt_path=CKPT_PATH,
    patience=PATIENCE, verbose=True
)

## 10. Training Curves

In [ ]:
epochs_ran = len(history['train_loss'])
ep = range(1, epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(ep, history['train_loss'], label='Train')
ax1.plot(ep, history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(ep, history['train_acc'], label='Train')
ax2.plot(ep, history['val_acc'],   label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.suptitle('Training History', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Evaluation on Validation Set

In [ ]:
# Load best checkpoint
final_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
final_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in vl_loader:
        images = images.to(DEVICE)
        outputs = final_model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Map indices back to original labels
all_preds_orig  = [IDX2LABEL[p] for p in all_preds]
all_labels_orig = [IDX2LABEL[l] for l in all_labels]

print('\nClassification Report:')
print(classification_report(all_labels_orig, all_preds_orig))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels_orig, all_preds_orig, labels=list(IDX2LABEL.values()))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(IDX2LABEL.values()),
            yticklabels=list(IDX2LABEL.values()), ax=ax)
ax.set_title('Confusion Matrix — Validation Set')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 12. Test-Time Augmentation (TTA)

TTA averages softmax probabilities over multiple augmented views of each test image (horizontal flip + original). This typically improves accuracy by 0.5–1.5% at the cost of 2× inference time.

In [ ]:
@torch.no_grad()
def predict_with_tta(model, dataset, device, batch_size=32, n_tta=2):
    """
    Generate predictions using Test-Time Augmentation.
    Averages logits over `n_tta` passes: original + h-flip.
    """
    tta_transforms = [
        val_transform,
        T.Compose([
            T.Resize(IMG_SIZE + 20),
            T.CenterCrop(IMG_SIZE),
            T.RandomHorizontalFlip(p=1.0),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    ]

    model.eval()
    all_ids    = []
    sum_logits = None

    for transform in tta_transforms[:n_tta]:
        dataset.transform = transform
        loader = DataLoader(dataset, batch_size=batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)
        batch_logits = []
        batch_ids    = []
        for images, ids in loader:
            images = images.to(device)
            logits = model(images)
            batch_logits.append(logits.cpu())
            batch_ids.extend(ids)
        logits_tensor = torch.cat(batch_logits, dim=0)
        if sum_logits is None:
            sum_logits = logits_tensor
            all_ids    = batch_ids
        else:
            sum_logits += logits_tensor

    avg_logits = sum_logits / n_tta
    _, preds   = avg_logits.max(1)
    return all_ids, preds.numpy()


print('TTA function defined ✓')

## 13. Generate Predictions for Test Set

In [ ]:
# Re-load best model
final_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
final_model = final_model.to(DEVICE)

# Test dataset (no labels)
test_id_col = [c for c in test_df.columns if 'id' in c.lower() or 'image' in c.lower()][0]
test_ds = ImageDataset(test_df, IMG_DIR, transform=val_transform,
                        id_col=test_id_col, label_col=None)
test_ds.label2idx = LABEL2IDX
test_ds.idx2label = IDX2LABEL

print('Generating predictions with TTA …')
test_ids, test_preds_idx = predict_with_tta(final_model, test_ds, DEVICE)
test_preds_labels = [IDX2LABEL[p] for p in test_preds_idx]

print(f'Predictions generated for {len(test_ids)} images.')
print('Sample predictions:', test_preds_labels[:10])

## 14. Save Submission CSV

In [ ]:
submission = test_df.copy()
submission['label'] = test_preds_labels
submission.to_csv(OUTPUT_CSV, index=False)
print(f'Saved predictions to: {OUTPUT_CSV}')
submission.head(10)

## 15. Summary & Key Design Decisions

| Component | Choice | Rationale |
|---|---|---|
| **Backbone** | EfficientNet-B2 (pretrained) | State-of-the-art accuracy/compute trade-off; ImageNet features transfer well |
| **Custom Dataset** | `ImageDataset(Dataset)` | Lazy loading, flexible extension resolution, supports train & test modes |
| **Augmentation** | RandomCrop, Flip, Rotation, ColorJitter, GaussianBlur, RandomErasing | Diverse transforms prevent overfitting; RandomErasing mimics occlusion |
| **Optimiser** | AdamW | Decoupled weight decay; typically outperforms Adam + L2 |
| **Scheduler** | CosineAnnealingLR | Smooth LR decay avoids sudden drops; shown to improve generalisation |
| **Loss** | CrossEntropy with label smoothing (0.1) | Reduces overconfidence; improves calibration |
| **HP Search** | Optuna TPE | Bayesian; more sample-efficient than grid/random search |
| **Training strategy** | Freeze → Unfreeze (2-stage) | Prevents catastrophic forgetting during early fine-tuning |
| **Early stopping** | patience=8 | Saves compute; avoids overfitting |
| **Inference** | Test-Time Augmentation (TTA) | Free accuracy boost at inference time |

### Possible Further Improvements
- **K-Fold cross-validation** ensemble: train K models, average logits at inference.
- **Larger backbone**: EfficientNet-B4/B5 or ConvNeXt-Base if compute allows.
- **Mixup / CutMix** augmentation for additional regularisation.
- **Stochastic Weight Averaging (SWA)** as a cheap ensembling alternative.
- **Progressive resizing**: train at lower resolution first, then fine-tune at full resolution.